In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import binomtest
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score, balanced_accuracy_score,
                             matthews_corrcoef, confusion_matrix, roc_auc_score, average_precision_score, roc_curve)
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

PATH = os.environ.get("HR_CSV", "HRDataset.csv")
OUT = os.environ.get("HR_OUT", "outputs")
EPOCHS = int(os.environ.get("EPOCHS", 50))
SEED = 42
CV_SEEDS = [int(s) for s in os.environ.get("CV_SEEDS", "42,123,256,512,1024").split(",") if s]
CV_FOLDS = int(os.environ.get("CV_FOLDS", 5))
ABL_SEEDS = [int(s) for s in os.environ.get("ABL_SEEDS", "42,123,256,512,1024").split(",") if s]
N_BOOT = int(os.environ.get("N_BOOT", 10000))
RUN_CV = int(os.environ.get("RUN_CV", 1))
RUN_ABLATION = int(os.environ.get("RUN_ABLATION", 1))
RUN_XAI = int(os.environ.get("RUN_XAI", 1))
XAI_CLASS = int(os.environ.get("XAI_CLASS", 1))
XAI_N = int(os.environ.get("XAI_N", 60))
CLASSES = ["Fully Meets", "Exceeds", "Needs Improvement", "PIP"]
DEV = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({"font.size": 12, "axes.labelcolor": "black", "xtick.color": "black", "ytick.color": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": "white"})


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def save_fig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, name), dpi=300, bbox_inches="tight")
    plt.close()


seed_all(SEED)

print("\n[STEP 1] Load data and audit row counts")
raw = pd.read_csv(PATH, encoding="latin1")
raw.columns = raw.columns.str.strip()
row_log = [("rows in file", len(raw))]
df = raw[raw["Performance Score"].isin(CLASSES)].copy()
row_log.append(("rows with one of the four labels", len(df)))
df = df.drop_duplicates()
row_log.append(("rows after drop_duplicates", len(df)))
print(pd.DataFrame(row_log, columns=["step", "rows"]).to_string(index=False))
counts = df["Performance Score"].value_counts().reindex(CLASSES)
print(pd.DataFrame({"count": counts, "percent": (100 * counts / len(df)).round(1)}))
print("unique Employee ID:", df["Employee ID"].nunique(), "of", len(df), "rows")
if df["Employee ID"].nunique() < len(df):
    print("WARNING: repeated employees, use a group split by Employee ID")
pd.DataFrame(row_log, columns=["step", "rows"]).to_csv(os.path.join(OUT, "row_audit.csv"), index=False)

print("\n[STEP 2] Leakage-safe features")
REF = pd.Timestamp("2024-01-01")
for c in ["StartDate", "DOB", "Survey Date", "Training Date"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
    print(c, "unparsed share:", round(df[c].isna().mean(), 3))
num_raw = ["Engagement Score", "Satisfaction Score", "Work-Life Balance Score", "Training Duration (Days)", "Training Cost"]
for c in num_raw:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")
df["tenure_days"] = (REF - df["StartDate"]).dt.days
df["age"] = (REF - df["DOB"]).dt.days / 365.25
df["training_recency_days"] = (REF - df["Training Date"]).dt.days
df["survey_recency_days"] = (REF - df["Survey Date"]).dt.days
df["eng_x_sat"] = df["Engagement Score"] * df["Satisfaction Score"]
df["wellbeing_composite"] = df[["Engagement Score", "Satisfaction Score", "Work-Life Balance Score"]].mean(axis=1)
df["training_efficiency"] = df["Training Duration (Days)"] / (df["Training Cost"] + 1)
df["seniority_ratio"] = df["tenure_days"] / (df["age"] * 365.25 + 1)
NUM = num_raw + ["tenure_days", "age", "training_recency_days", "survey_recency_days", "eng_x_sat",
                 "wellbeing_composite", "training_efficiency", "seniority_ratio"]
CAT = ["DepartmentType", "PayZone", "GenderCode", "Training Type", "Training Program Name", "BusinessUnit", "Division",
       "JobFunctionDescription", "EmployeeClassificationType", "MaritalDesc", "LocationCode"]
FEATS = NUM + CAT
EXCLUDED = ["Current Employee Rating", "TerminationType", "TerminationDescription", "ExitDate", "EmployeeStatus"]
assert not set(EXCLUDED) & set(FEATS)
assert len(FEATS) == 24
N_NUM = len(NUM)
y_all = df["Performance Score"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
gender_all = df["GenderCode"].astype(str).to_numpy()
Xdf = df[FEATS].reset_index(drop=True)
print("features used:", len(FEATS), "| excluded:", EXCLUDED)

print("\n[STEP 3] Stratified 70/15/15 split")
all_idx = np.arange(len(Xdf))
i_tr, i_tmp = train_test_split(all_idx, test_size=0.30, stratify=y_all, random_state=SEED)
i_va, i_te = train_test_split(i_tmp, test_size=0.50, stratify=y_all[i_tmp], random_state=SEED)
split_table = pd.DataFrame({"train": np.bincount(y_all[i_tr], minlength=4), "val": np.bincount(y_all[i_va], minlength=4),
                            "test": np.bincount(y_all[i_te], minlength=4)}, index=CLASSES)
split_table.loc["Total"] = split_table.sum()
print(split_table)
split_table.to_csv(os.path.join(OUT, "table_split.csv"))

print("\n[STEP 4] Preprocessing fitted on training rows only")


def build(fit_idx):
    tr = Xdf.iloc[fit_idx].copy()
    tr[NUM] = tr[NUM].replace([np.inf, -np.inf], np.nan)
    med = tr[NUM].median()
    mode = tr[CAT].mode().iloc[0]

    def prep(p):
        p = p.copy()
        p[NUM] = p[NUM].replace([np.inf, -np.inf], np.nan).fillna(med)
        p[CAT] = p[CAT].fillna(mode).astype(str)
        return p

    trp = prep(tr)
    lo, hi = trp[NUM].quantile(0.01), trp[NUM].quantile(0.99)
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(trp[CAT])
    sc = StandardScaler().fit(trp[NUM].clip(lo, hi, axis=1))
    ctr = enc.transform(trp[CAT]).astype(int) + 1
    mu, sd = ctr.mean(0), ctr.std(0) + 1e-6
    card = [int(ctr[:, j].max()) + 2 for j in range(len(CAT))]

    def transform(ix):
        p = prep(Xdf.iloc[ix])
        xn = sc.transform(p[NUM].clip(lo, hi, axis=1))
        xc = enc.transform(p[CAT]).astype(int) + 1
        return np.hstack([xn, (xc - mu) / sd]).astype(np.float32), xc.astype(np.int64)

    return transform, card, (mu, sd)


transform, card, (mu, sd) = build(i_tr)
Xs_tr, Xc_tr = transform(i_tr)
Xs_va, Xc_va = transform(i_va)
Xs_te, Xc_te = transform(i_te)
y_tr, y_va, y_te = y_all[i_tr], y_all[i_va], y_all[i_te]
g_te = gender_all[i_te]
print("train", Xs_tr.shape, "val", Xs_va.shape, "test", Xs_te.shape, "| NaN in train features:", int(np.isnan(Xs_tr).sum()))

print("\n[STEP 5] Model definitions")


def onehot(xc, cards):
    return np.hstack([np.eye(c, dtype=np.float32)[xc[:, j]] for j, c in enumerate(cards)])


def fit_conventional(name, Xs, Xc, y, Xs_t, Xc_t, cards, seed=SEED):
    tree_tr = np.hstack([Xs[:, :N_NUM], Xc])
    tree_te = np.hstack([Xs_t[:, :N_NUM], Xc_t])
    sw = compute_sample_weight("balanced", y)
    if name == "LogReg":
        m = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0)
        m.fit(np.hstack([Xs[:, :N_NUM], onehot(Xc, cards)]), y)
        return m.predict_proba(np.hstack([Xs_t[:, :N_NUM], onehot(Xc_t, cards)]))
    if name == "RF":
        m = RandomForestClassifier(500, class_weight="balanced", n_jobs=-1, random_state=seed).fit(tree_tr, y)
        return m.predict_proba(tree_te)
    if name == "XGBoost":
        m = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                          random_state=seed, n_jobs=-1, eval_metric="mlogloss")
        m.fit(tree_tr, y, sample_weight=sw)
        return m.predict_proba(tree_te)
    m = CatBoostClassifier(iterations=500, learning_rate=0.03, depth=6, auto_class_weights="Balanced",
                           random_seed=seed, verbose=0, thread_count=-1).fit(tree_tr, y)
    return m.predict_proba(tree_te)


class Tokenizer(nn.Module):
    def __init__(self, n_num, cards, d):
        super().__init__()
        self.n_num = n_num
        self.w = nn.Parameter(torch.randn(n_num, d) * 0.02)
        self.b = nn.Parameter(torch.zeros(n_num, d))
        self.emb = nn.ModuleList([nn.Embedding(c, d) for c in cards])

    def forward(self, xs, xc):
        tn = xs[:, :self.n_num].unsqueeze(-1) * self.w + self.b
        tc = torch.stack([e(xc[:, j]) for j, e in enumerate(self.emb)], 1)
        return torch.cat([tn, tc], 1)


class Encoder(nn.Module):
    def __init__(self, d, heads, layers, drop):
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, batch_first=True, norm_first=True, activation="gelu")
        self.tf = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, tok):
        x = torch.cat([self.cls.expand(tok.size(0), -1, -1), tok], 1)
        return self.norm(self.tf(x)[:, 0])


class MLP(nn.Module):
    def __init__(self, n_num, cards, h=256, drop=0.2, n_cls=4):
        super().__init__()
        self.n_num, self.cards = n_num, cards
        self.net = nn.Sequential(nn.Linear(n_num + sum(cards), h), nn.ReLU(), nn.Dropout(drop),
                                 nn.Linear(h, h // 2), nn.ReLU(), nn.Dropout(drop), nn.Linear(h // 2, n_cls))

    def forward(self, xs, xc):
        oh = torch.cat([Fn.one_hot(xc[:, j], c).float() for j, c in enumerate(self.cards)], 1)
        return self.net(torch.cat([xs[:, :self.n_num], oh], 1))


class TabTrans(nn.Module):
    def __init__(self, n_num, cards, d=64, heads=4, layers=2, drop=0.1, n_cls=4):
        super().__init__()
        self.n_num = n_num
        self.emb = nn.ModuleList([nn.Embedding(c, d) for c in cards])
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, batch_first=True, norm_first=True, activation="gelu")
        self.tf = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.ln = nn.LayerNorm(n_num)
        self.mlp = nn.Sequential(nn.Linear(len(cards) * d + n_num, 256), nn.GELU(), nn.Dropout(drop), nn.Linear(256, n_cls))

    def forward(self, xs, xc):
        t = torch.stack([e(xc[:, j]) for j, e in enumerate(self.emb)], 1)
        return self.mlp(torch.cat([self.tf(t).flatten(1), self.ln(xs[:, :self.n_num])], 1))


class FTT(nn.Module):
    def __init__(self, n_num, cards, d=64, heads=4, layers=3, drop=0.1, n_cls=4):
        super().__init__()
        self.tok = Tokenizer(n_num, cards, d)
        self.enc = Encoder(d, heads, layers, drop)
        self.head = nn.Linear(d, n_cls)

    def forward(self, xs, xc):
        return self.head(self.enc(self.tok(xs, xc)))


class Selector(nn.Module):
    def __init__(self, F, steps=3, na=32, nd=32, gamma=1.3):
        super().__init__()
        self.T, self.na, self.gamma = steps, na, gamma
        self.bn0 = nn.BatchNorm1d(F)
        self.init = nn.Linear(F, na)
        self.att = nn.ModuleList([nn.Linear(na, F) for _ in range(steps)])
        self.bna = nn.ModuleList([nn.BatchNorm1d(F) for _ in range(steps)])
        self.feat = nn.ModuleList([nn.Linear(F, na + nd) for _ in range(steps)])
        self.bnf = nn.ModuleList([nn.BatchNorm1d(na + nd) for _ in range(steps)])

    def forward(self, x):
        x = self.bn0(x)
        P = torch.ones_like(x)
        h = torch.relu(self.init(x))
        out = 0
        masks = []
        for t in range(self.T):
            M = torch.softmax(P * self.bna[t](self.att[t](h)), dim=-1)
            f = self.bnf[t](self.feat[t](M * x))
            h, d = torch.relu(f[:, :self.na]), torch.relu(f[:, self.na:])
            out = out + d
            P = P * (self.gamma - M)
            masks.append(M)
        return out, torch.stack(masks, 1).mean(1)


class TabNetClf(nn.Module):
    def __init__(self, F, nd=32, n_cls=4):
        super().__init__()
        self.sel = Selector(F, 3, 32, nd)
        self.head = nn.Sequential(nn.Linear(nd, 64), nn.ReLU(), nn.Dropout(0.1), nn.Linear(64, n_cls))

    def forward(self, xs, xc):
        return self.head(self.sel(xs)[0])


class TTSE(nn.Module):
    def __init__(self, n_num, cards, d=64, heads=4, layers=3, drop=0.1, nd=32, n_cls=4, sgb="full", fusion=True):
        super().__init__()
        F = n_num + len(cards)
        self.sgb, self.fusion = sgb, fusion
        self.sel = Selector(F, 3, 32, nd)
        self.tok = Tokenizer(n_num, cards, d)
        self.enc = Encoder(d, heads, layers, drop)
        self.gate = nn.Linear(F, F)
        self.generic = nn.Parameter(torch.zeros(F))
        self.head = nn.Sequential(nn.Linear((nd if fusion else 0) + d, 256), nn.GELU(), nn.Dropout(drop),
                                  nn.Linear(256, 128), nn.GELU(), nn.Dropout(drop), nn.Linear(128, n_cls))

    def forward(self, xs, xc, return_mask=False):
        z_tab, M = self.sel(xs)
        tok = self.tok(xs, xc)
        if self.sgb == "full":
            G = torch.sigmoid(self.gate(M))
        elif self.sgb == "direct":
            G = M * M.size(1)
        elif self.sgb == "generic":
            G = torch.sigmoid(self.generic).expand(xs.size(0), -1)
        else:
            G = None
        if G is not None:
            tok = tok * G.unsqueeze(-1)
        z = self.enc(tok)
        z = torch.cat([z_tab, z], 1) if self.fusion else z
        out = self.head(z)
        return (out, M) if return_mask else out


MAKERS = {
    "MLP": (lambda c: MLP(N_NUM, c), 1e-3),
    "TabTransformer": (lambda c: TabTrans(N_NUM, c), 3e-4),
    "FT-Transformer": (lambda c: FTT(N_NUM, c), 1e-4),
    "TabNet": (lambda c: TabNetClf(N_NUM + len(c)), 5e-3),
    "TTSE": (lambda c: TTSE(N_NUM, c), 2e-4),
}
CONVENTIONAL = ["LogReg", "RF", "XGBoost", "CatBoost"]


def run_dl(make, cards, Xs, Xc, y, Xs_v, Xc_v, y_v, Xs_t, Xc_t, seed, lr, epochs=EPOCHS, bs=64, wd=1e-4, patience=10):
    seed_all(seed)
    m = make(cards).to(DEV)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    wt = torch.tensor(len(y) / (4 * np.maximum(np.bincount(y, minlength=4), 1)), dtype=torch.float32)
    dl = DataLoader(TensorDataset(torch.tensor(Xs), torch.tensor(Xc), torch.tensor(y)), batch_size=bs, shuffle=True, drop_last=True)

    def predict(a, b):
        m.eval()
        o = []
        with torch.no_grad():
            for i in range(0, len(a), 512):
                o.append(m(torch.tensor(a[i:i + 512]).to(DEV), torch.tensor(b[i:i + 512]).to(DEV)).cpu())
        return torch.cat(o)

    best, bad, state = 1e9, 0, None
    hist = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}
    for ep in range(epochs):
        m.train()
        tl = tc = n = 0
        for xs, xc, yb in dl:
            xs, xc, yb = xs.to(DEV), xc.to(DEV), yb.to(DEV)
            opt.zero_grad()
            out = m(xs, xc)
            loss = Fn.cross_entropy(out, yb, weight=wt.to(DEV))
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(yb)
            tc += (out.argmax(1) == yb).sum().item()
            n += len(yb)
        sch.step()
        lg = predict(Xs_v, Xc_v)
        yv = torch.tensor(y_v)
        vl = Fn.cross_entropy(lg, yv, weight=wt).item()
        va = (lg.argmax(1) == yv).float().mean().item()
        for k, v in zip(hist, (tl / n, vl, tc / n, va)):
            hist[k].append(v)
        if vl < best - 1e-5:
            best, bad = vl, 0
            state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    m.load_state_dict(state)
    return torch.softmax(predict(Xs_t, Xc_t), 1).numpy(), hist, m


print("\n[STEP 6] Train every model on the 70/15/15 split and store test predictions")
probs, hists, trained = {}, {}, {}
for k in CONVENTIONAL:
    probs[k] = fit_conventional(k, Xs_tr, Xc_tr, y_tr, Xs_te, Xc_te, card)
    print(f"{k:15s} test accuracy {100 * (probs[k].argmax(1) == y_te).mean():.2f}%")
for k, (mk, lr) in MAKERS.items():
    probs[k], hists[k], trained[k] = run_dl(mk, card, Xs_tr, Xc_tr, y_tr, Xs_va, Xc_va, y_va, Xs_te, Xc_te, SEED, lr)
    print(f"{k:15s} test accuracy {100 * (probs[k].argmax(1) == y_te).mean():.2f}% | final validation accuracy "
          f"{100 * hists[k]['va_acc'][-1]:.2f}% | epochs run {len(hists[k]['va_acc'])}")
pred_df = pd.DataFrame({"id": np.arange(len(y_te)), "y_true": y_te, "gender": g_te})
for k, p in probs.items():
    pred_df[f"pred_{k}"] = p.argmax(1)
    for c in range(4):
        pred_df[f"prob_{k}_{c}"] = p[:, c]
pred_df.to_csv(os.path.join(OUT, "test_predictions.csv"), index=False)
pd.DataFrame({k: v for k, v in hists["TTSE"].items()}).to_csv(os.path.join(OUT, "ttse_training_history.csv"), index=False)

print("\n[STEP 7] Table 12 from stored predictions")
P = pd.read_csv(os.path.join(OUT, "test_predictions.csv"))
y = P["y_true"].to_numpy()
n = len(P)
models = [c[5:] for c in P.columns if c.startswith("pred_")]
rows = []
for m in models:
    pm = P[f"pred_{m}"].to_numpy()
    mp, mr, mf, _ = precision_recall_fscore_support(y, pm, average="macro", zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y, pm, average="weighted", zero_division=0)
    ip, ir, iff, _ = precision_recall_fscore_support(y, pm, average="micro", zero_division=0)
    rows.append({"Model": m, "Accuracy": 100 * accuracy_score(y, pm), "Micro-P=R=F1": 100 * iff,
                 "Macro-P": 100 * mp, "Macro-R": 100 * mr, "Macro-F1": 100 * mf,
                 "Weighted-P": 100 * wp, "Weighted-R": 100 * wr, "Weighted-F1": 100 * wf,
                 "BalancedAcc": 100 * balanced_accuracy_score(y, pm), "MCC": matthews_corrcoef(y, pm)})
table12 = pd.DataFrame(rows).round(2)
print(table12.to_string(index=False))
table12.to_csv(os.path.join(OUT, "table12.csv"), index=False)

print("\n[STEP 8] TTSE confusion matrix and class-wise table")
pt = P["pred_TTSE"].to_numpy()
cm = confusion_matrix(y, pt, labels=[0, 1, 2, 3])
tp, sup, pred_n = np.diag(cm), cm.sum(1), cm.sum(0)
prec = tp / np.maximum(pred_n, 1)
rec = tp / sup
f1 = 2 * tp / np.maximum(pred_n + sup, 1)


def wilson_text(k, m_):
    lo, hi = proportion_confint(k, m_, alpha=0.05, method="wilson")
    return f"[{100 * lo:.1f}, {100 * hi:.1f}]"


class_table = pd.DataFrame({"Class": CLASSES, "Support": sup, "TP": tp, "Predicted as class": pred_n,
                            "Precision %": (100 * prec).round(2), "Precision Wilson CI": [wilson_text(tp[i], pred_n[i]) for i in range(4)],
                            "Recall %": (100 * rec).round(2), "Recall Wilson CI": [wilson_text(tp[i], sup[i]) for i in range(4)],
                            "F1 %": (100 * f1).round(2)})
print(cm)
print(class_table.to_string(index=False))
print("correct:", int(tp.sum()), "of", n, "| accuracy", round(100 * tp.sum() / n, 2))
print("macro P/R/F1:", round(100 * prec.mean(), 2), round(100 * rec.mean(), 2), round(100 * f1.mean(), 2),
      "| weighted F1:", round(100 * (f1 * sup).sum() / sup.sum(), 2))
class_table.to_csv(os.path.join(OUT, "table13_class_wise.csv"), index=False)
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(os.path.join(OUT, "confusion_matrix_ttse.csv"))
fig, ax = plt.subplots(figsize=(5.6, 4.8))
im = ax.imshow(cm, cmap="Greens")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=12, color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(CLASSES, rotation=25, ha="right", color="black")
ax.set_yticklabels(CLASSES, color="black")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title(f"TTSE confusion matrix (n = {n}, accuracy = {100 * tp.sum() / n:.2f}%)", fontsize=12, pad=10)
plt.colorbar(im, ax=ax, fraction=0.046)
save_fig("fig_confusion_matrix.png")

print("\n[STEP 9] McNemar exact test, Holm adjustment and paired bootstrap")
rng = np.random.default_rng(SEED)
boot_idx = rng.integers(0, n, size=(N_BOOT, n))
ref = P["pred_TTSE"].to_numpy() == y
res_rows = []
for m in [x for x in models if x != "TTSE"]:
    other = P[f"pred_{m}"].to_numpy() == y
    b = int((ref & ~other).sum())
    c = int((~ref & other).sum())
    p_exact = binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0
    d = ref.astype(float) - other.astype(float)
    boot = d[boot_idx].mean(axis=1)
    res_rows.append({"Comparator": m, "b TTSE-only correct": b, "c comparator-only correct": c,
                     "Acc diff % = (b-c)/n": 100 * (b - c) / n, "p exact": p_exact, "Boot mean %": 100 * boot.mean(),
                     "CI low %": 100 * np.percentile(boot, 2.5), "CI high %": 100 * np.percentile(boot, 97.5)})
mcn = pd.DataFrame(res_rows)
mcn["p Holm"] = multipletests(mcn["p exact"], method="holm")[1]
print(mcn.round(4).to_string(index=False))
mcn.to_csv(os.path.join(OUT, "tables16_17_paired_tests.csv"), index=False)

print("\n[STEP 10] Gender subgroup analysis from integer counts")


def wil(k, m_):
    return proportion_confint(k, m_, alpha=0.05, method="wilson") if m_ > 0 else (np.nan, np.nan)


def newcombe(k1, n1, k2, n2):
    if n1 == 0 or n2 == 0:
        return np.nan, np.nan, np.nan
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = wil(k1, n1)
    l2, u2 = wil(k2, n2)
    d = p1 - p2
    return d, d - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2), d + np.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)


def pct(k, m_):
    return 100 * k / m_ if m_ else np.nan


genders = sorted(set(P["gender"]))
assert len(genders) == 2
ga, gb = genders
g = P["gender"].to_numpy()
fair_rows = []
for ci, cname in enumerate(CLASSES):
    r = {"Class": cname}
    for gv in (ga, gb):
        pos = (y == ci) & (g == gv)
        neg = (y != ci) & (g == gv)
        r[f"n_pos_{gv}"] = int(pos.sum())
        r[f"TP_{gv}"] = int((pt[pos] == ci).sum())
        r[f"n_neg_{gv}"] = int(neg.sum())
        r[f"FP_{gv}"] = int((pt[neg] == ci).sum())
    d, lo, hi = newcombe(r[f"TP_{ga}"], r[f"n_pos_{ga}"], r[f"TP_{gb}"], r[f"n_pos_{gb}"])
    fd, flo, fhi = newcombe(r[f"FP_{ga}"], r[f"n_neg_{ga}"], r[f"FP_{gb}"], r[f"n_neg_{gb}"])
    r.update({f"Recall_{ga} %": pct(r[f"TP_{ga}"], r[f"n_pos_{ga}"]), f"Recall_{gb} %": pct(r[f"TP_{gb}"], r[f"n_pos_{gb}"]),
              "EqOpp diff": d, "EqOpp CI low": lo, "EqOpp CI high": hi,
              f"FPR_{ga} %": pct(r[f"FP_{ga}"], r[f"n_neg_{ga}"]), f"FPR_{gb} %": pct(r[f"FP_{gb}"], r[f"n_neg_{gb}"]),
              "FPR diff": fd, "FPR CI low": flo, "FPR CI high": fhi})
    if r[f"n_pos_{ga}"] and r[f"n_pos_{gb}"]:
        overall = (r[f"TP_{ga}"] + r[f"TP_{gb}"]) / (r[f"n_pos_{ga}"] + r[f"n_pos_{gb}"])
        lo_r, hi_r = sorted([r[f"TP_{ga}"] / r[f"n_pos_{ga}"], r[f"TP_{gb}"] / r[f"n_pos_{gb}"]])
        assert lo_r - 1e-9 <= overall <= hi_r + 1e-9
    fair_rows.append(r)
fair = pd.DataFrame(fair_rows)
print("test employees per gender:", dict(P["gender"].value_counts()))
print(fair.round(3).T.to_string())
fair.to_csv(os.path.join(OUT, "tables18_19_fairness.csv"), index=False)

print("\n[STEP 11] ROC-AUC, PR-AUC and ECE from stored probabilities")
Y = label_binarize(y, classes=[0, 1, 2, 3])


def ece_score(Pm, yy, bins=10):
    conf, pred = Pm.max(1), Pm.argmax(1)
    acc = (pred == yy).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mk = (conf > lo) & (conf <= hi)
        if mk.any():
            e += mk.mean() * abs(acc[mk].mean() - conf[mk].mean())
    return e


auc_rows = []
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for m in models:
    Pm = P[[f"prob_{m}_{c}" for c in range(4)]].to_numpy()
    roc = roc_auc_score(Y, Pm, average="macro")
    auc_rows.append({"Model": m, "ROC-AUC macro OvR": roc, "PR-AUC macro": average_precision_score(Y, Pm, average="macro"),
                     "ECE": ece_score(Pm, y)})
    fprs, tprs = zip(*[roc_curve(Y[:, c], Pm[:, c])[:2] for c in range(4)])
    grid = np.unique(np.concatenate(fprs))
    ax.plot(grid, np.mean([np.interp(grid, f, t) for f, t in zip(fprs, tprs)], axis=0), lw=2.4 if m == "TTSE" else 1.4,
            label=f"{m} (AUC = {roc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.legend(fontsize=8, loc="lower right")
save_fig("fig_roc_all_models.png")
auc_table = pd.DataFrame(auc_rows).round(4)
print(auc_table.to_string(index=False))
auc_table.to_csv(os.path.join(OUT, "table_auc_ece.csv"), index=False)

print("\n[STEP 12] Training and validation curves of TTSE")
h = hists["TTSE"]
ep = np.arange(1, len(h["tr_loss"]) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(ep, h["tr_loss"], label="Training loss")
a1.plot(ep, h["va_loss"], "--", label="Validation loss")
a1.set_xlabel("Epoch")
a1.set_ylabel("Loss")
a1.legend()
a2.plot(ep, 100 * np.array(h["tr_acc"]), label="Training accuracy")
a2.plot(ep, 100 * np.array(h["va_acc"]), "--", label="Validation accuracy")
a2.axhline(100 * (pt == y).mean(), color="red", ls=":", label="Test accuracy (held-out)")
a2.set_xlabel("Epoch")
a2.set_ylabel("Accuracy (%)")
a2.legend()
save_fig("fig_ttse_curves.png")
print("final training acc %.2f | final validation acc %.2f | test acc %.2f" % (100 * h["tr_acc"][-1], 100 * h["va_acc"][-1], 100 * (pt == y).mean()))

if RUN_CV:
    print("\n[STEP 13] Repeated stratified cross-validation on the non-test records")
    pool = np.concatenate([i_tr, i_va])
    print("records in CV pool:", len(pool), "| test records excluded:", len(i_te))
    cv_rows = []
    for name in CONVENTIONAL + list(MAKERS):
        for s in CV_SEEDS:
            skf = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=s)
            for fold, (a, b) in enumerate(skf.split(pool, y_all[pool])):
                ia, ib = pool[a], pool[b]
                tf, cd, _ = build(ia)
                Xa, Ca = tf(ia)
                Xb, Cb = tf(ib)
                ya, yb = y_all[ia], y_all[ib]
                if name in CONVENTIONAL:
                    p = fit_conventional(name, Xa, Ca, ya, Xb, Cb, cd, s)
                else:
                    j, l = train_test_split(np.arange(len(ia)), test_size=0.1, stratify=ya, random_state=s)
                    mk, lr = MAKERS[name]
                    p, _, _ = run_dl(mk, cd, Xa[j], Ca[j], ya[j], Xa[l], Ca[l], ya[l], Xb, Cb, s, lr)
                pr = p.argmax(1)
                cv_rows.append({"Model": name, "seed": s, "fold": fold, "Accuracy": 100 * accuracy_score(yb, pr),
                                "Macro-F1": 100 * precision_recall_fscore_support(yb, pr, average="macro", zero_division=0)[2]})
        print("finished", name)
    cv = pd.DataFrame(cv_rows)
    cv.to_csv(os.path.join(OUT, "cv_all_runs.csv"), index=False)
    cv_summary = cv.groupby("Model")[["Accuracy", "Macro-F1"]].agg(["mean", "std"]).round(2)
    print(cv_summary)
    cv_summary.to_csv(os.path.join(OUT, "cv_summary.csv"))

if RUN_ABLATION:
    print("\n[STEP 14] Ablation of the Selective Gating Bridge")
    variants = {"TTSE full": dict(sgb="full", fusion=True), "w/o SGB": dict(sgb="none", fusion=True),
                "w/o fusion": dict(sgb="full", fusion=False), "Direct mask": dict(sgb="direct", fusion=True),
                "Generic gate": dict(sgb="generic", fusion=True)}
    abl_rows = []
    abl_pred = pd.DataFrame({"id": np.arange(len(y_te)), "y_true": y_te})
    for vname, kw in variants.items():
        for s in ABL_SEEDS:
            p, _, _ = run_dl(lambda c, kw=kw: TTSE(N_NUM, c, **kw), card, Xs_tr, Xc_tr, y_tr, Xs_va, Xc_va, y_va, Xs_te, Xc_te, s, 2e-4)
            pr = p.argmax(1)
            abl_rows.append({"Variant": vname, "seed": s, "Accuracy": 100 * accuracy_score(y_te, pr),
                             "Macro-F1": 100 * precision_recall_fscore_support(y_te, pr, average="macro", zero_division=0)[2]})
            abl_pred[f"pred_{vname}_seed{s}"] = pr
    abl = pd.DataFrame(abl_rows)
    abl_summary = abl.groupby("Variant", sort=False)[["Accuracy", "Macro-F1"]].agg(["mean", "std"]).round(2)
    print(abl_summary)
    abl.to_csv(os.path.join(OUT, "ablation_all_runs.csv"), index=False)
    abl_summary.to_csv(os.path.join(OUT, "ablation_summary.csv"))
    abl_pred.to_csv(os.path.join(OUT, "ablation_predictions.csv"), index=False)

if RUN_XAI:
    print("\n[STEP 15] SHAP and LIME for TTSE")
    import shap
    from lime.lime_tabular import LimeTabularExplainer
    model = trained["TTSE"]
    Z_tr = np.hstack([Xs_tr[:, :N_NUM], Xc_tr]).astype(np.float32)
    Z_te = np.hstack([Xs_te[:, :N_NUM], Xc_te]).astype(np.float32)
    card_arr = np.array(card)

    def predict_raw(Z):
        Z = np.asarray(Z, dtype=np.float32)
        xc = np.minimum(np.clip(np.rint(Z[:, N_NUM:]), 0, None).astype(np.int64), card_arr - 1)
        xs = np.hstack([Z[:, :N_NUM], (xc - mu) / sd]).astype(np.float32)
        model.eval()
        with torch.no_grad():
            lg = model(torch.tensor(xs).to(DEV), torch.tensor(xc).to(DEV))
        return torch.softmax(lg, 1).cpu().numpy()

    background = shap.sample(Z_tr, 50, random_state=SEED)
    explainer = shap.KernelExplainer(predict_raw, background)
    Zx = Z_te[:XAI_N]
    sv = explainer.shap_values(Zx, nsamples=200)
    sv = np.asarray(sv)
    vals = sv[XAI_CLASS] if isinstance(sv, list) or (sv.ndim == 3 and sv.shape[0] == 4) else sv[:, :, XAI_CLASS]
    base = np.atleast_1d(explainer.expected_value)[XAI_CLASS]
    shap.summary_plot(vals, Zx, feature_names=FEATS, show=False)
    save_fig("fig_shap_beeswarm.png")
    shap.decision_plot(base, vals[:20], feature_names=FEATS, show=False)
    save_fig("fig_shap_decision.png")
    mean_abs = pd.Series(np.abs(vals).mean(0), index=FEATS).sort_values(ascending=False)
    print(mean_abs.head(10).round(4))
    mean_abs.to_csv(os.path.join(OUT, "shap_mean_abs.csv"))
    lime_exp = LimeTabularExplainer(Z_tr, feature_names=FEATS, class_names=CLASSES, mode="classification",
                                    categorical_features=list(range(N_NUM, len(FEATS))), random_state=SEED)
    exp = lime_exp.explain_instance(Z_te[0], predict_raw, num_features=12, labels=(XAI_CLASS,))
    exp.as_pyplot_figure(label=XAI_CLASS)
    save_fig("fig_lime_bar.png")

print("\nDone. Every table and figure is in:", OUT)